# original excel to schema migration

In [ ]:
import pandas as pd
import numpy as np

### add phi default for null positions to IPA tables

In [ ]:
#consonantIPA = pd.read_csv('../tables/consonantIPA.csv')

In [ ]:
# insert row at index 0 with column values as specified
# consonantIPA.loc[-1] = [0.0, '∅', np.nan, 0, 0, 0, 0, 0, 'B', 3, '∅', 0.0]
# consonantIPA.sort_values(by = 'IPA_key', inplace=True)
# consonantIPA.reset_index(drop=True, inplace=True)

In [ ]:
# assign datatypes
#consonantIPA['consonantal'] = consonantIPA['consonantal'].astype(bool)
#consonantIPA['voice'] = consonantIPA['voice'].astype(bool)
#consonantIPA['sibilant'] = consonantIPA['sibilant'].astype(bool)
#consonantIPA['lateral'] = consonantIPA['lateral'].astype(bool)
#consonantIPA['geminate'] = consonantIPA['geminate'].astype(bool)


In [ ]:
#consonantIPA.to_csv('../tables/consonantIPA.csv', index=False)

In [ ]:
#vowelIPA = pd.read_csv('../tables/vowelIPA.csv')

In [ ]:
# insert row at index 0 with column values as specified for phi
# vowelIPA.loc[-1] = [0.0, '∅', 0, 0, 0, 0, 0, 'Ə', 3.0, 'Ɵ', 0.0]
# vowelIPA.sort_values(by = 'IPA_key', inplace=True)
# vowelIPA.reset_index(drop=True, inplace=True)

In [ ]:
# assign correct dtypes to binary cols
#vowelIPA['rounded'] = vowelIPA['rounded'].astype(bool)
#vowelIPA['nasal'] = vowelIPA['nasal'].astype(bool)
#vowelIPA['front'] = vowelIPA['front'].astype(bool)
#vowelIPA['raised'] = vowelIPA['raised'].astype(bool)
#vowelIPA['retracted'] = vowelIPA['retracted'].astype(bool)

In [ ]:
#vowelIPA.to_csv('../tables/vowelIPA.csv', index=False)

### trtmtEnv from all_consonants_data

In [ ]:
acd = pd.read_csv('all_consonants_data.csv')

In [ ]:
# get existing treatment and environment combinations
trtmtEnv = acd[['treatment', 'environment', 'number']].drop_duplicates()

trtmtEnv.head(10)


In [ ]:
type(acd['number'][0])

In [ ]:
# try to coerce .0 decimals to ints
trtmtEnv.rename(columns={'number': 'trtmt_env_id'}, inplace=True)

def coerce_id(x):
    try:
        f = float(x)
        return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return x.lstrip('0') or '0'

trtmtEnv['trtmt_env_id'] = trtmtEnv['trtmt_env_id'].apply(coerce_id)

#### parse original excel for ref words and sound changes

In [ ]:
# parse json (copilot created)
import json
with open(r"overview_sound_changes_filtered_per_sheet.json", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
# convert to dataframe
rows = []
for key, value in data.items():
    id_part, treatment_part = key.split(" ", 1)

    if id_part.isdigit():
        id_part = int(id_part)
        id_part = str(id_part)
    else:
        # strip leading 0
        id_part = id_part.lstrip("0")
    rows.append({
        "trtmt_env_id": id_part,
        "treatment": treatment_part,
        "sound_changes": value,
    })

df = pd.DataFrame(rows, columns=["trtmt_env_id", "treatment", "sound_changes"])
df['trtmt_env_id'] = df['trtmt_env_id'].apply(coerce_id)
df["treatment"] = df["treatment"].astype(str)

In [ ]:
# join to trtmt Env
trtmtEnv = pd.merge(trtmtEnv, df, on=["trtmt_env_id", "treatment"], how = 'outer')

In [ ]:
## now do same for ref words
with open(r"latin_reference_words_per_sheet.json", encoding="utf-8") as f:
    ref_words = json.load(f)

In [ ]:
# convert to dataframe
rows = []
for key, value in ref_words.items():
    id_part, treatment_part = key.split(" ", 1)

    if id_part.isdigit():
        id_part = float(id_part)
    else:
        # strip leading 0
        id_part = id_part.lstrip("0")
    rows.append({
        "trtmt_env_id": id_part,
        "treatment": treatment_part,
        "reference_words": value,
    })

df = pd.DataFrame(rows, columns=["trtmt_env_id", "treatment", "reference_words"])
df['trtmt_env_id'] = df['trtmt_env_id'].apply(coerce_id)

df["treatment"] = df["treatment"].astype(str)

trtmtEnv = pd.merge(trtmtEnv, df, on=["trtmt_env_id", "treatment"], how = 'outer')

In [ ]:
trtmtEnv.to_csv('../tables/trtmtEnv.csv', index=False)

### trtmtEnvSegments

approaching from raw xlsx instead of csv to mitigate debugging errors (aka did the mistake come from the originally processed data or the new code)\
going to modify original excel_to_tables code

* get as is, then go back through and parse components

In [ ]:
excel_file = pd.ExcelFile('QDFH.xlsx')

sheet_names = excel_file.sheet_names
data_sheets = sheet_names[1:-4]

singles = np.r_[0:31, 55:77, 93:len(data_sheets)]
singles = [data_sheets[i] for i in singles]

doubles = np.r_[31:47, 77:93]
doubles = [data_sheets[i] for i in doubles]

with_vowels = np.r_[47:55]


#### single consonants

In [ ]:
## single consonants


# single consonant treatments
single_treatment_tables = []

for sheet_name in singles: #[0:33]

        df = excel_file.parse(sheet_name) 
        print(f"Sheet: {sheet_name}")

        # get basic info
        number = df.iloc[1,1]
        treatment = df.iloc[2,1]
        environment = df.iloc[3, 1]
        
        table = df.iloc[5:17, :]
        
        # pivot table so rows = cols
        melted = table.melt(id_vars = 'A - Consonantism', var_name = 'Variable', value_name = 'Value')
        pivot = melted.pivot(index = 'Variable', columns = 'A - Consonantism', values = 'Value')
        pivot.set_index('Languages', inplace = True)
        pivot.reset_index(inplace = True)

        pivot = pivot.dropna(subset = ['Languages'])


        # deal with a/b/c versions
        pivot['version'] = pivot.groupby('Languages').cumcount().add(1)
        pivot['version'] = pivot['version'].apply(lambda x:  chr(97 + int(x)-1))

        pivot.insert(0, 'trtmt_env_id', number)
        pivot.insert(0, 'treatment', treatment)
        pivot.insert(1, 'environment', environment)
        pivot.insert(2, 'position', 1)
        pivot.insert(3, 'is_vowel', False)

        # clean language names and empty rows
        pivot['Languages'] = pivot['Languages'].str.replace('\n', ' ', regex = False)
        
        pivot.rename(columns = {'Languages': 'language', 'IPA key': 'IPA_key'}, inplace = True)
        pivot = pivot[['trtmt_env_id', 'language', 'version', 'position', 'IPA_key', 'IPA', 'is_vowel']]

        single_treatment_tables.append(pivot)

all_single_tables = pd.concat(single_treatment_tables, axis=0)

#### double consonants

In [ ]:

double_treatment_tables = []
n_consonants = 2

for sheet_name in doubles: 

        df = excel_file.parse(sheet_name) 
        print(f"Sheet: {sheet_name}")


        # get basic info
        number = df.iloc[1,1]
        treatment = df.iloc[2,1]
        environment = df.iloc[3, 1]

        treatment_tables = []
        
        for i in range(0, n_consonants): 
                if i == 0:
                        pos_table = df.iloc[5:17, :]
                elif i == 1:
                        pos_table = df.iloc[np.r_[5, 18:29], :]
                

                # pivot table so rows = cols
                melted =  pos_table.melt(id_vars = 'A - Consonantism', var_name = 'Variable', value_name = 'Value')
                pivot = melted.pivot(index = 'Variable', columns = 'A - Consonantism', values = 'Value')
                pivot.set_index('Languages', inplace = True)
                pivot.reset_index(inplace = True)

                pivot = pivot.dropna(subset = ['Languages'])

                # deal with a/b/c versions
                pivot['version'] = pivot.groupby('Languages').cumcount().add(1)
                pivot['version'] = pivot['version'].apply(lambda x:  chr(97 + int(x)-1))

                pivot.insert(0, 'trtmt_env_id', number)
                pivot.insert(0, 'treatment', treatment)
                pivot.insert(1, 'environment', environment)
                pivot.insert(2, 'position', i+1)
                pivot.insert(3, 'is_vowel', False)

                # clean language names and empty rows
                pivot['Languages'] = pivot['Languages'].str.replace('\n', ' ', regex = False)

                pivot.rename(columns = {'Languages': 'language', 'IPA key': 'IPA_key'}, inplace = True)
                pivot = pivot[['trtmt_env_id', 'language', 'version', 'position', 'IPA_key', 'IPA', 'is_vowel']]

                treatment_tables.append(pivot)
        
        double_treatment_table = pd.concat(treatment_tables, axis = 0)
        double_treatment_table.sort_values(['language'], inplace =True)
        double_treatment_tables.append(double_treatment_table)

all_double_tables = pd.concat(double_treatment_tables, axis = 0)

#### with vowels

In [ ]:
with_vowels_list = [data_sheets[i] for i in with_vowels]

vowel_treatment_tables = []

for sheet_name in with_vowels_list:

    df = excel_file.parse(sheet_name)
    print(f"Sheet: {sheet_name}")

    # get basic info
    number = df.iloc[1, 1]
    treatment = df.iloc[2, 1]
    environment = df.iloc[3, 1]

    # count consonants from treatment name (e.g. "SP-" -> 2, "SCR-" -> 3)
    n_consonants = len([c for c in str(treatment) if c.isalpha()])

    treatment_tables = []

    # --- Vowel section: rows 5:20 (Languages header + 14 vowel feature rows) ---
    vowel_table = df.iloc[5:20, :]

    melted = vowel_table.melt(id_vars='A - Consonantism', var_name='Variable', value_name='Value')
    pivot = melted.pivot(index='Variable', columns='A - Consonantism', values='Value')
    pivot.set_index('Languages', inplace=True)
    pivot.reset_index(inplace=True)
    pivot = pivot.dropna(subset=['Languages'])

    pivot['version'] = pivot.groupby('Languages').cumcount().add(1)
    pivot['version'] = pivot['version'].apply(lambda x: chr(97 + int(x) - 1))

    pivot.insert(0, 'trtmt_env_id', number)
    pivot.insert(0, 'treatment', treatment)
    pivot.insert(1, 'environment', environment)
    pivot.insert(2, 'position', 0)
    pivot.insert(3, 'is_vowel', True)

    pivot['Languages'] = pivot['Languages'].str.replace('\n', ' ', regex=False)
    pivot.rename(columns={'Languages': 'language', 'IPA key': 'IPA_key'}, inplace=True)
    pivot = pivot[['trtmt_env_id', 'language', 'version', 'position', 'IPA_key', 'IPA', 'is_vowel']]

    treatment_tables.append(pivot)

    # --- Consonant sections ---
    # First consonant data starts at row 21 (after vowel rows 5-19 + blank at 20)
    # Each section: 11 data rows + 1 blank = 12 row stride
    for i in range(n_consonants):
        start_row = 21 + i * 12
        pos_table = df.iloc[np.r_[5, start_row:start_row + 11], :]

        melted = pos_table.melt(id_vars='A - Consonantism', var_name='Variable', value_name='Value')
        pivot = melted.pivot(index='Variable', columns='A - Consonantism', values='Value')
        pivot.set_index('Languages', inplace=True)
        pivot.reset_index(inplace=True)
        pivot = pivot.dropna(subset=['Languages'])

        pivot['version'] = pivot.groupby('Languages').cumcount().add(1)
        pivot['version'] = pivot['version'].apply(lambda x: chr(97 + int(x) - 1))

        pivot.insert(0, 'trtmt_env_id', number)
        pivot.insert(0, 'treatment', treatment)
        pivot.insert(1, 'environment', environment)
        pivot.insert(2, 'position', i + 1)
        pivot.insert(3, 'is_vowel', False)

        pivot['Languages'] = pivot['Languages'].str.replace('\n', ' ', regex=False)
        pivot.rename(columns={'Languages': 'language', 'IPA key': 'IPA_key'}, inplace=True)
        pivot = pivot[['trtmt_env_id', 'language', 'version', 'position', 'IPA_key', 'IPA', 'is_vowel']]

        treatment_tables.append(pivot)

    vowel_treatment_table = pd.concat(treatment_tables, axis=0)
    vowel_treatment_table.sort_values(['language'], inplace=True)
    vowel_treatment_tables.append(vowel_treatment_table)

all_vowel_tables = pd.concat(vowel_treatment_tables, axis=0)

In [ ]:
## combine all tables
trtmtEnvSegments = pd.concat([all_single_tables, all_double_tables, all_vowel_tables], axis=0)
trtmtEnvSegments.dropna(subset = ['IPA'], inplace=True)

In [ ]:

import re

# Load IPA lookup tables and combine into one IPA -> IPA_key mapping
vowel_ipa_lookup = pd.read_csv('../tables/vowelIPA.csv')[['IPA_key', 'IPA']]
consonant_ipa_lookup = pd.read_csv('../tables/consonantIPA.csv')[['IPA_key', 'IPA']]
ipa_lookup = pd.concat([vowel_ipa_lookup, consonant_ipa_lookup], axis=0).drop_duplicates(subset=['IPA'])


def split_ipa_row(row):
    """
    Split a row's IPA value into component rows.
    - '~' separator (outside parentheses): equal main_usage_prop, is_dialect=False for all parts
    - '(...)' parenthesis: main form is_dialect=False prop=1,
                           parenthesised variant is_dialect=True prop=0
                           if parenthesised value contains '~', take only the first element
    - component: 'a', 'b', 'c', ... identifies each split part
    Returns a list of dicts, one per component.
    """
    ipa = str(row['IPA']) if pd.notna(row['IPA']) else ''
    base = {col: row[col] for col in row.index if col not in ['IPA', 'IPA_key']}

    # Check for '~' only outside parentheses
    if '~' in re.sub(r'\(.*?\)', '', ipa):
        parts = [p.strip() for p in ipa.split('~')]
        prop = round(1 / len(parts), 6)
        return [{**base, 'IPA': p, 'component': chr(97 + i), 'is_dialect': False, 'main_usage_prop': prop}
                for i, p in enumerate(parts)]

    elif '(' in ipa:
        # Remove parenthesised part to get main IPA, but keep it for dialect variant
        main = re.sub(r'\s*\(.*?\)', '', ipa).strip()
        # Now check if the parenthesised part contains '~' and take only the first element if so
        dialect_match = re.search(r'\((.*?)\)', ipa)
        result = [{**base, 'IPA': main, 'component': 'a', 'is_dialect': False, 'main_usage_prop': 1}]
        if dialect_match:
            dialect = dialect_match.group(1).strip()
            if '~' in dialect:
                dialect = dialect.split('~')[0].strip()
            result.append({**base, 'IPA': dialect, 'component': 'b', 'is_dialect': True, 'main_usage_prop': 0})
        return result

    else:
        return [{**base, 'IPA': ipa, 'component': 'a', 'is_dialect': False, 'main_usage_prop': 1}]


# Expand rows
expanded = []
for _, row in trtmtEnvSegments.iterrows():
    expanded.extend(split_ipa_row(row))

trtmtEnvSegments_parsed = pd.DataFrame(expanded)

# Re-join IPA_key from lookup tables based on split IPA values
trtmtEnvSegments_parsed = trtmtEnvSegments_parsed.merge(
    ipa_lookup.rename(columns={'IPA_key': 'IPA_key_new'}),
    on='IPA',
    how='left'
).rename(columns={'IPA_key': 'IPA_key_old', 'IPA_key_new': 'IPA_key'})

# Reorder columns
col_order = ['trtmt_env_id', 'language', 'version', 'position', 'IPA_key', 'IPA',
             'is_vowel', 'component', 'is_dialect', 'main_usage_prop']
trtmtEnvSegments_parsed = trtmtEnvSegments_parsed[col_order]


In [ ]:
trtmtEnvSegments_parsed[trtmtEnvSegments_parsed['IPA_key'].isna()]

In [ ]:
trtmtEnvSegments_parsed.to_csv('../tables/trtmtEnvSegments.csv', index=False)
